# Part 4: Best Model Training and Sample Generation


In [ ]:
!pip install tokenizers torch lxml cairosvg matplotlib tqdm

## 1. Imports and Configuration

In [ ]:
import json
import math
import os
import random
import time
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, List

import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
from lxml import etree
from tokenizers import Tokenizer
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from tqdm import tqdm

# Paths
TOKENIZER_PATH     = Path('svg_tokenizer_6.json')
TRAIN_TOKENS_PATH  = Path('train_tokens_6.pt')
VAL_TOKENS_PATH    = Path('val_tokens_6.pt')
TEST_TOKENS_PATH   = Path('test_tokens_6.pt')
PART3_RESULTS_PATH = Path('part3_mup_results/part3_results.pt')
PART3_LR_PATH      = Path('part3_mup_results/best_lr.txt')

OUTPUT_DIR  = Path('part4_outputs')
SAMPLES_DIR = OUTPUT_DIR / 'samples'
RENDER_DIR  = OUTPUT_DIR / 'rendered'
METRICS_PATH = OUTPUT_DIR / 'metrics.json'
MODEL_PATH  = OUTPUT_DIR / 'best_model.pt'

for d in [SAMPLES_DIR, RENDER_DIR]:
    d.mkdir(parents=True, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

## 2. Load Tokenizer and Data

In [ ]:
for p in [TOKENIZER_PATH, TRAIN_TOKENS_PATH, VAL_TOKENS_PATH]:
    if not p.exists():
        raise FileNotFoundError(f'Missing required file: {p}  — run Part 1 first.')

tokenizer    = Tokenizer.from_file(str(TOKENIZER_PATH))
train_tokens = torch.load(str(TRAIN_TOKENS_PATH), weights_only=False)
val_tokens   = torch.load(str(VAL_TOKENS_PATH),   weights_only=False)
test_tokens  = torch.load(str(TEST_TOKENS_PATH),  weights_only=False) if TEST_TOKENS_PATH.exists() else None

PAD_ID = tokenizer.token_to_id('<pad>')
BOS_ID = tokenizer.token_to_id('<bos>')
EOS_ID = tokenizer.token_to_id('<eos>')

total_train_tokens = sum(len(s) for s in train_tokens)
print(f'Vocab size         : {tokenizer.get_vocab_size():,}')
print(f'Train sequences    : {len(train_tokens):,}  ({total_train_tokens/1e6:.1f}M tokens)')
print(f'Val   sequences    : {len(val_tokens):,}')
if test_tokens:
    print(f'Test  sequences    : {len(test_tokens):,}')
print(f'BOS={BOS_ID}  EOS={EOS_ID}  PAD={PAD_ID}')

## 3. Model Architecture

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 1024):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pos   = torch.arange(max_len).unsqueeze(1).float()
        div   = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe    = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(1)].unsqueeze(0)
        return self.dropout(x)


class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        assert d_model % n_heads == 0
        self.n_heads  = n_heads
        self.head_dim = d_model // n_heads
        self.qkv      = nn.Linear(d_model, 3 * d_model)
        self.out_proj = nn.Linear(d_model, d_model)
        self.dropout  = nn.Dropout(dropout)
        self.d_model  = d_model

    def forward(self, x, mask=None):
        B, T, C = x.shape
        qkv = self.qkv(x).view(B, T, 3, self.n_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        Q, K, V = qkv[0], qkv[1], qkv[2]
        scale   = math.sqrt(self.head_dim)
        scores  = torch.matmul(Q, K.transpose(-2, -1)) / scale
        if mask is not None:
            scores = scores.masked_fill(~mask, float('-inf'))
        attn  = torch.softmax(scores, dim=-1)
        out   = torch.matmul(self.dropout(attn), V)
        out   = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.out_proj(out)


class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.attn  = MultiHeadSelfAttention(d_model, n_heads, dropout)
        self.ff    = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.drop  = nn.Dropout(dropout)

    def forward(self, x, mask):
        # Pre-norm architecture (more stable at larger scales)
        x = x + self.drop(self.attn(self.norm1(x), mask))
        x = x + self.drop(self.ff(self.norm2(x)))
        return x


class DecoderOnlyTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff,
                 dropout=0.1, max_len=1024, padding_idx=None):
        super().__init__()
        self.embed   = nn.Embedding(vocab_size, d_model, padding_idx=padding_idx)
        self.pos_enc = PositionalEncoding(d_model, dropout, max_len)
        self.layers  = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        self.norm    = nn.LayerNorm(d_model)
        self.head    = nn.Linear(d_model, vocab_size, bias=False)
        # Tie input/output embeddings (reduces parameters, often improves perplexity)
        self.head.weight = self.embed.weight
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def _causal_mask(self, sz, device):
        return torch.tril(torch.ones(sz, sz, device=device, dtype=torch.bool))

    def forward(self, src):
        mask = self._causal_mask(src.size(1), src.device)
        x = self.embed(src) * math.sqrt(self.embed.embedding_dim)
        x = self.pos_enc(x)
        for layer in self.layers:
            x = layer(x, mask)
        return self.head(self.norm(x))

    def count_params(self):
        return sum(p.numel() for p in self.parameters())


MODEL_CONFIGS = {
    'tiny'  : {'d_model': 128, 'n_layers': 6,  'n_heads': 4,  'd_ff': 512},
    'small' : {'d_model': 192, 'n_layers': 6,  'n_heads': 6,  'd_ff': 768},
    'medium': {'d_model': 384, 'n_layers': 6,  'n_heads': 12, 'd_ff': 1536},
    'large' : {'d_model': 512, 'n_layers': 6,  'n_heads': 16, 'd_ff': 2048},
    'xl'    : {'d_model': 768, 'n_layers': 6,  'n_heads': 24, 'd_ff': 3072},
}

print('Model configs loaded.')

## 4. Training Utilities

In [ ]:
class TokenDataset:
    def __init__(self, sequences: List[List[int]]):
        # Store as flat list for fast random access
        self.data = sequences

    def sample_batch(self, batch_size: int, block_size: int, device: str):
        x_list, y_list = [], []
        # Filter to only sequences long enough
        eligible = [s for s in self.data if len(s) > block_size]
        chosen   = random.choices(eligible, k=batch_size)
        for seq in chosen:
            start = random.randint(0, len(seq) - block_size - 1)
            chunk = seq[start: start + block_size + 1]
            x_list.append(torch.tensor(chunk[:-1], dtype=torch.long))
            y_list.append(torch.tensor(chunk[1:],  dtype=torch.long))
        return torch.stack(x_list).to(device), torch.stack(y_list).to(device)


def get_scheduler(optimizer, total_steps, warmup_steps):
    warmup = LinearLR(optimizer, start_factor=1e-6, end_factor=1.0,
                      total_iters=max(1, warmup_steps))
    cosine = CosineAnnealingLR(optimizer, T_max=max(1, total_steps - warmup_steps))
    return SequentialLR(optimizer, [warmup, cosine], milestones=[warmup_steps])


def steps_per_epoch(total_tokens, batch_size, block_size):
    return max(1, math.ceil(total_tokens / (batch_size * block_size)))


@torch.no_grad()
def evaluate(model, dataset, device, steps=50, batch_size=8, block_size=512):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    total = 0.0
    for _ in range(steps):
        x, y = dataset.sample_batch(batch_size, block_size, device)
        logits = model(x)
        total += criterion(logits.view(-1, logits.size(-1)), y.view(-1)).item()
    return total / steps


def train_one_epoch(model, dataset, optimizer, scheduler, device,
                    steps, batch_size, block_size, log_interval=100):
    model.train()
    criterion  = nn.CrossEntropyLoss()
    loss_log   = []
    running    = 0.0
    t0         = time.time()

    for step in range(1, steps + 1):
        x, y   = dataset.sample_batch(batch_size, block_size, device)
        logits = model(x)
        loss   = criterion(logits.view(-1, logits.size(-1)), y.view(-1))

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        running += loss.item()
        if step % log_interval == 0 or step == steps:
            avg = running / log_interval
            loss_log.append(avg)
            print(f'  Step {step:>5}/{steps}  loss={avg:.4f}  '
                  f'lr={scheduler.get_last_lr()[0]:.2e}')
            running = 0.0

    elapsed     = time.time() - t0
    throughput  = steps * batch_size * block_size / elapsed
    return loss_log, elapsed, throughput


print('Training utilities defined.')

## 5. Generation Utilities

In [ ]:
def top_k_top_p_filter(logits, top_k=0, top_p=0.0):
    """Apply top-k and/or nucleus (top-p) filtering to a logits vector."""
    if top_k > 0:
        threshold = torch.topk(logits, min(top_k, logits.size(-1)))[0][..., -1, None]
        logits    = logits.masked_fill(logits < threshold, float('-inf'))

    if 0.0 < top_p < 1.0:
        sorted_logits, sorted_idx = torch.sort(logits, descending=True)
        cum_probs = sorted_logits.softmax(dim=-1).cumsum(dim=-1)
        # Remove tokens with cumulative probability above top_p
        # (keep at least the highest-probability token)
        remove_mask        = cum_probs > top_p
        remove_mask[..., 1:] = remove_mask[..., :-1].clone()
        remove_mask[..., 0]  = False
        logits[sorted_idx[remove_mask]] = float('-inf')

    return logits


@torch.no_grad()
def generate(
    model,
    prefix_ids: List[int],
    max_new_tokens: int = 256,
    temperature: float = 1.0,
    top_k: int = 40,
    top_p: float = 0.9,
    eos_id: int = None,
    block_size: int = 512,
):
    """
    Autoregressive generation with temperature / top-k / top-p sampling.
    Stops early when <eos> is emitted.
    """
    model.eval()
    generated = list(prefix_ids)

    for _ in range(max_new_tokens):
        context = torch.tensor([generated[-block_size:]], dtype=torch.long, device=device)
        logits  = model(context)[0, -1, :]  # (vocab_size,)

        # Temperature scaling
        logits = logits / max(temperature, 1e-8)
        logits = top_k_top_p_filter(logits, top_k=top_k, top_p=top_p)

        probs      = logits.softmax(dim=-1)
        next_token = torch.multinomial(probs, num_samples=1).item()
        generated.append(next_token)

        if eos_id is not None and next_token == eos_id:
            break

    return generated


def decode(token_ids: List[int]) -> str:
    """Decode token IDs to string, stripping special tokens."""
    # Remove BOS/EOS/PAD
    special = {BOS_ID, EOS_ID, PAD_ID}
    ids = [t for t in token_ids if t not in special]
    return tokenizer.decode(ids)


def encode_prefix(text: str) -> List[int]:
    """Encode a text prefix, prepending <bos>."""
    return [BOS_ID] + tokenizer.encode(text).ids


print('Generation utilities defined.')

## 6. Evaluation Utilities

In [ ]:
import cairosvg

def is_xml_valid(svg_text: str) -> bool:
    """Check whether the generated text is well-formed XML with an <svg> root."""
    try:
        root = etree.fromstring(svg_text.encode('utf-8'))
        tag  = root.tag
        return tag == 'svg' or tag.endswith('}svg')
    except Exception:
        return False


def is_structurally_valid(svg_text: str) -> bool:
    """Check for <svg> root, proper closure, and at least one shape element."""
    if not is_xml_valid(svg_text):
        return False
    has_shape = bool(re.search(r'<(path|circle|rect|polygon|ellipse|line)', svg_text))
    return has_shape


def render_to_png(svg_text: str, out_path: Path, size: int = 256) -> bool:
    """Attempt to render SVG to PNG. Returns True on success."""
    try:
        cairosvg.svg2png(bytestring=svg_text.encode('utf-8'),
                         write_to=str(out_path),
                         output_width=size, output_height=size)
        return True
    except Exception:
        return False


def compute_metrics(samples: List[dict]) -> dict:
    """Aggregate validity metrics over a list of sample dicts."""
    n = len(samples)
    if n == 0:
        return {}
    xml_valid      = sum(s['xml_valid']         for s in samples)
    struct_valid   = sum(s['struct_valid']       for s in samples)
    rendered       = sum(s['rendered']           for s in samples)
    return {
        'n'                   : n,
        'xml_valid_rate'      : xml_valid    / n,
        'struct_valid_rate'   : struct_valid / n,
        'render_rate'         : rendered     / n,
    }


import re
print('Evaluation utilities defined.')

## 7. Select Best Model from Parts 2/3

In [ ]:
def load_best_lr(default: float = 2e-4) -> float:
    if PART3_LR_PATH.exists():
        try:
            return float(PART3_LR_PATH.read_text().strip())
        except Exception:
            pass
    return default


def select_best_model_name() -> str:
    if PART3_RESULTS_PATH.exists():
        try:
            data    = torch.load(PART3_RESULTS_PATH, weights_only=False)
            entries = data.get('mup_results', data.get('results', []))
            valid   = [(e['name'], e['val_loss']) for e in entries
                       if 'name' in e and 'val_loss' in e]
            if valid:
                best, loss = min(valid, key=lambda x: x[1])
                print(f'Auto-selected from Part 3 results: {best} (val_loss={loss:.4f})')
                return best
        except Exception as e:
            print(f'Could not read Part 3 results: {e}')
    print('No Part 3 results found — defaulting to "large".')
    return 'large'


# ---- Configuration ----
MODEL_NAME = select_best_model_name()       # override: MODEL_NAME = 'xl'
BEST_LR    = load_best_lr()
EPOCHS     = 2      # increase if resources allow
BATCH_SIZE = 16
BLOCK_SIZE = 512
DROPOUT    = 0.1
WEIGHT_DECAY = 0.01

# Generation settings
MAX_NEW_TOKENS = 300
TEMPERATURES   = [0.5, 0.8, 1.0]
TOP_K          = 40
TOP_P          = 0.9

print(f'Model name : {MODEL_NAME}')
print(f'Learning rate: {BEST_LR:.2e}')
print(f'Epochs     : {EPOCHS}')
print(f'Batch size : {BATCH_SIZE}')
print(f'Block size : {BLOCK_SIZE}')

## 8. Build Model

In [ ]:
cfg = MODEL_CONFIGS[MODEL_NAME]

model = DecoderOnlyTransformer(
    vocab_size   = tokenizer.get_vocab_size(),
    d_model      = cfg['d_model'],
    n_layers     = cfg['n_layers'],
    n_heads      = cfg['n_heads'],
    d_ff         = cfg['d_ff'],
    dropout      = DROPOUT,
    max_len      = BLOCK_SIZE,
    padding_idx  = PAD_ID,
).to(device)

n_params = model.count_params()
print(f'Model : {MODEL_NAME}  ({n_params/1e6:.2f}M parameters)')
print(f'Config: {cfg}')

# Print parameter breakdown
for name, p in model.named_parameters():
    if p.requires_grad:
        print(f'  {name:50s}  {list(p.shape)}  ({p.numel()/1e3:.1f}K)')


## 9. Train

In [ ]:
train_ds = TokenDataset(train_tokens)
val_ds   = TokenDataset(val_tokens)
test_ds  = TokenDataset(test_tokens) if test_tokens else None

epoch_steps  = steps_per_epoch(total_train_tokens, BATCH_SIZE, BLOCK_SIZE)
warmup_steps = max(1, epoch_steps // 5)
total_steps  = epoch_steps * EPOCHS

optimizer = optim.AdamW(model.parameters(), lr=BEST_LR,
                         betas=(0.9, 0.95), weight_decay=WEIGHT_DECAY)
scheduler = get_scheduler(optimizer, total_steps, warmup_steps)

print(f'Steps per epoch : {epoch_steps:,}')
print(f'Warmup steps    : {warmup_steps:,}')
print(f'Total steps     : {total_steps:,}')

In [ ]:
all_train_losses = []
val_loss_history = []

if MODEL_PATH.exists():
    print(f'Checkpoint found at {MODEL_PATH} — loading.')
    model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
else:
    for epoch in range(1, EPOCHS + 1):
        print(f'\n=== Epoch {epoch}/{EPOCHS} ===')
        losses, elapsed, tput = train_one_epoch(
            model, train_ds, optimizer, scheduler, device,
            epoch_steps, BATCH_SIZE, BLOCK_SIZE, log_interval=200
        )
        val_loss = evaluate(model, val_ds, device, steps=50,
                            batch_size=8, block_size=BLOCK_SIZE)
        all_train_losses.extend(losses)
        val_loss_history.append(val_loss)
        print(f'  Val loss={val_loss:.4f}  ppl={math.exp(val_loss):.2f}  '
              f'elapsed={elapsed:.0f}s  {tput/1e3:.1f}K tok/s')

    torch.save(model.state_dict(), MODEL_PATH)
    print(f'\nModel saved to {MODEL_PATH}')

In [ ]:
# Final evaluation
val_loss  = evaluate(model, val_ds, device, steps=100, batch_size=8, block_size=BLOCK_SIZE)
val_ppl   = math.exp(val_loss)
print(f'Validation loss : {val_loss:.4f}  |  Perplexity : {val_ppl:.2f}')

if test_ds:
    test_loss = evaluate(model, test_ds, device, steps=100, batch_size=8, block_size=BLOCK_SIZE)
    test_ppl  = math.exp(test_loss)
    print(f'Test loss       : {test_loss:.4f}  |  Perplexity : {test_ppl:.2f}')
else:
    test_loss = test_ppl = None

In [ ]:
if all_train_losses:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(all_train_losses, label='train loss')
    axes[0].set_xlabel('Log steps (×log_interval)')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('Training loss')
    axes[0].legend()

    if val_loss_history:
        axes[1].plot(range(1, len(val_loss_history)+1), val_loss_history,
                     marker='o', label='val loss')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Loss')
        axes[1].set_title('Validation loss per epoch')
        axes[1].legend()

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'training_curves.png', dpi=150)
    plt.show()

## 10. Unconditional Sample Generation

In [ ]:
N_UNCONDITIONAL = 10
uncond_results  = []

print('Generating unconditional samples...\n')
for idx in range(N_UNCONDITIONAL):
    temp     = TEMPERATURES[idx % len(TEMPERATURES)]
    prefix   = encode_prefix('<svg ')  # BOS + start of SVG

    gen_ids  = generate(
        model, prefix,
        max_new_tokens=MAX_NEW_TOKENS,
        temperature=temp, top_k=TOP_K, top_p=TOP_P,
        eos_id=EOS_ID, block_size=BLOCK_SIZE
    )
    svg_text = decode(gen_ids)

    svg_name = f'uncond_{idx+1:02d}_t{temp:.1f}.svg'
    svg_path = SAMPLES_DIR / svg_name
    png_path = RENDER_DIR  / svg_name.replace('.svg', '.png')
    svg_path.write_text(svg_text, encoding='utf-8')

    xml_ok    = is_xml_valid(svg_text)
    struct_ok = is_structurally_valid(svg_text)
    rendered  = render_to_png(svg_text, png_path) if xml_ok else False

    uncond_results.append({
        'idx': idx+1, 'temperature': temp,
        'svg_path': str(svg_path), 'png_path': str(png_path),
        'xml_valid': xml_ok, 'struct_valid': struct_ok, 'rendered': rendered,
        'n_tokens': len(gen_ids),
    })

    print(f'  [{idx+1:02d}] temp={temp}  xml={xml_ok}  struct={struct_ok}  '
          f'rendered={rendered}  tokens={len(gen_ids)}')

uncond_metrics = compute_metrics(uncond_results)
print(f'\nUnconditional metrics:')
for k, v in uncond_metrics.items():
    print(f'  {k}: {v:.3f}' if isinstance(v, float) else f'  {k}: {v}')

## 11. Prefix-Conditioned Generation

In [ ]:
PREFIX_EXAMPLES = [
    {
        'name'  : 'partial_face',
        'desc'  : 'Left eye + pupil — does the model add the right eye and mouth?',
        'prefix': '<svg viewBox="0 0 64 64" xmlns="http://www.w3.org/2000/svg">'
                  '<circle cx="20" cy="24" r="8" stroke="black" fill="none"/>'
                  '<circle cx="20" cy="24" r="3" fill="black"/>',
    },
    {
        'name'  : 'open_path',
        'desc'  : 'Unclosed L-shaped path — does the model close it and complete the shape?',
        'prefix': '<svg viewBox="0 0 64 64" xmlns="http://www.w3.org/2000/svg">'
                  '<path d="M10 10 L54 10 L54 54" stroke="black" fill="none"',
    },
    {
        'name'  : 'group_one_shape',
        'desc'  : 'Group with one rect — does the model add complementary shapes?',
        'prefix': '<svg viewBox="0 0 64 64" xmlns="http://www.w3.org/2000/svg">'
                  '<g><rect x="12" y="12" width="40" height="20" fill="#1f77b4"/>',
    },
    {
        'name'  : 'circle_divided',
        'desc'  : 'Circle with a vertical line — does the model complete it as a pie/dial?',
        'prefix': '<svg viewBox="0 0 64 64" xmlns="http://www.w3.org/2000/svg">'
                  '<circle cx="32" cy="32" r="18" fill="none" stroke="black"/>'
                  '<path d="M32 14 L32 50" stroke="black" fill="none"/>',
    },
    {
        'name'  : 'rounded_rect_icon',
        'desc'  : 'Card outline — does the model add icon content inside?',
        'prefix': '<svg viewBox="0 0 64 64" xmlns="http://www.w3.org/2000/svg">'
                  '<rect x="8" y="16" width="48" height="32" rx="6" '
                  'fill="none" stroke="black" stroke-width="2"/>',
    },
]

prefix_results = []
print('Generating prefix-conditioned samples...\n')

for ex in PREFIX_EXAMPLES:
    print(f"  Prefix '{ex['name']}'  — {ex['desc']}")
    for temp in TEMPERATURES:
        prefix_ids = encode_prefix(ex['prefix'])
        gen_ids    = generate(
            model, prefix_ids,
            max_new_tokens=MAX_NEW_TOKENS,
            temperature=temp, top_k=TOP_K, top_p=TOP_P,
            eos_id=EOS_ID, block_size=BLOCK_SIZE
        )
        svg_text = decode(gen_ids)

        svg_name = f"prefix_{ex['name']}_t{temp:.1f}.svg"
        svg_path = SAMPLES_DIR / svg_name
        png_path = RENDER_DIR  / svg_name.replace('.svg', '.png')
        svg_path.write_text(svg_text, encoding='utf-8')

        xml_ok    = is_xml_valid(svg_text)
        struct_ok = is_structurally_valid(svg_text)
        rendered  = render_to_png(svg_text, png_path) if xml_ok else False

        prefix_results.append({
            'name': ex['name'], 'temperature': temp,
            'prefix': ex['prefix'],
            'svg_path': str(svg_path), 'png_path': str(png_path),
            'xml_valid': xml_ok, 'struct_valid': struct_ok, 'rendered': rendered,
            'n_tokens': len(gen_ids),
        })
        print(f'    temp={temp}  xml={xml_ok}  struct={struct_ok}  '
              f'rendered={rendered}  tokens={len(gen_ids)}')

prefix_metrics = compute_metrics(prefix_results)
print(f'\nPrefix-conditioned metrics:')
for k, v in prefix_metrics.items():
    print(f'  {k}: {v:.3f}' if isinstance(v, float) else f'  {k}: {v}')

## 12. Sample Grid Visualization

In [ ]:
def render_grid(png_paths, titles, filename, ncols=5, cell_size=2.5):
    """Render a grid of PNG images."""
    valid = [(p, t) for p, t in zip(png_paths, titles) if Path(p).exists()]
    if not valid:
        print('No rendered PNGs to display.')
        return

    n      = len(valid)
    ncols  = min(ncols, n)
    nrows  = math.ceil(n / ncols)
    fig, axes = plt.subplots(nrows, ncols,
                             figsize=(ncols * cell_size, nrows * (cell_size + 0.4)))
    axes_flat = np.array(axes).flatten()

    for i, ax in enumerate(axes_flat):
        ax.axis('off')
        if i < n:
            img = mpimg.imread(valid[i][0])
            ax.imshow(img)
            ax.set_title(valid[i][1], fontsize=7)

    plt.tight_layout()
    plt.savefig(filename, dpi=200, bbox_inches='tight')
    plt.show()
    print(f'Grid saved to {filename}')


# Unconditional grid
uncond_pngs   = [r['png_path'] for r in uncond_results]
uncond_titles = [f"t={r['temperature']}" for r in uncond_results]
render_grid(uncond_pngs, uncond_titles,
            OUTPUT_DIR / 'uncond_sample_grid.png', ncols=5)

# Prefix-conditioned grid (one per prefix, best temp)
best_prefix = [
    next(r for r in prefix_results if r['name'] == ex['name'] and r['temperature'] == 0.8)
    for ex in PREFIX_EXAMPLES
]
render_grid(
    [r['png_path'] for r in best_prefix],
    [r['name']     for r in best_prefix],
    OUTPUT_DIR / 'prefix_sample_grid.png', ncols=5
)

## 13. Temperature Comparison

In [ ]:
# Show the same prefix at each temperature side by side
ref_example = PREFIX_EXAMPLES[0]  # partial_face
temp_rows   = [r for r in prefix_results if r['name'] == ref_example['name']]

fig, axes = plt.subplots(1, len(temp_rows), figsize=(3.5 * len(temp_rows), 4))
if len(temp_rows) == 1:
    axes = [axes]

for ax, row in zip(axes, temp_rows):
    ax.axis('off')
    pth = Path(row['png_path'])
    if pth.exists():
        ax.imshow(mpimg.imread(str(pth)))
    ax.set_title(
        f"T={row['temperature']}\nvalid={row['xml_valid']}  rendered={row['rendered']}",
        fontsize=9
    )

plt.suptitle(f"Prefix: '{ref_example['name']}' at different temperatures", y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'temperature_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 14. Save Metrics

In [ ]:
metrics = {
    'model'           : MODEL_NAME,
    'config'          : cfg,
    'params_M'        : n_params / 1e6,
    'val_loss'        : val_loss,
    'val_perplexity'  : val_ppl,
    'test_loss'       : test_loss,
    'test_perplexity' : test_ppl,
    'unconditional_metrics' : uncond_metrics,
    'prefix_metrics'        : prefix_metrics,
    'unconditional_samples' : uncond_results,
    'prefix_samples'        : prefix_results,
    'training': {
        'epochs'       : EPOCHS,
        'batch_size'   : BATCH_SIZE,
        'block_size'   : BLOCK_SIZE,
        'learning_rate': BEST_LR,
        'weight_decay' : WEIGHT_DECAY,
        'dropout'      : DROPOUT,
    },
    'sampling': {
        'temperatures'   : TEMPERATURES,
        'top_k'          : TOP_K,
        'top_p'          : TOP_P,
        'max_new_tokens' : MAX_NEW_TOKENS,
    },
}

with open(METRICS_PATH, 'w') as f:
    json.dump(metrics, f, indent=2, default=str)

print(f'Metrics saved to {METRICS_PATH}')

# Print summary table
print()
print('=' * 50)
print('PART 4 SUMMARY')
print('=' * 50)
print(f'Model              : {MODEL_NAME}  ({n_params/1e6:.2f}M params)')
print(f'Validation loss    : {val_loss:.4f}')
print(f'Validation PPL     : {val_ppl:.2f}')
if test_loss:
    print(f'Test loss          : {test_loss:.4f}')
    print(f'Test PPL           : {test_ppl:.2f}')
print()
print('Unconditional samples:')
print(f'  XML valid rate   : {uncond_metrics["xml_valid_rate"]*100:.1f}%')
print(f'  Struct valid rate: {uncond_metrics["struct_valid_rate"]*100:.1f}%')
print(f'  Render rate      : {uncond_metrics["render_rate"]*100:.1f}%')
print()
print('Prefix-conditioned samples:')
print(f'  XML valid rate   : {prefix_metrics["xml_valid_rate"]*100:.1f}%')
print(f'  Struct valid rate: {prefix_metrics["struct_valid_rate"]*100:.1f}%')
print(f'  Render rate      : {prefix_metrics["render_rate"]*100:.1f}%')

## 15. Qualitative Analysis (for report)



In [ ]:
# Show a few sample SVG strings for manual inspection
print('=== Sample unconditional SVG outputs ===\n')
for r in uncond_results[:3]:
    txt = Path(r['svg_path']).read_text(encoding='utf-8')
    print(f"-- Sample {r['idx']}  temp={r['temperature']}  "
          f"xml_valid={r['xml_valid']}  rendered={r['rendered']}")
    print(txt[:400] + ('...' if len(txt) > 400 else ''))
    print()